## Example 1 (Linear): 1D Hovercraft

In [ ]:
## Activate Project Environment
using Pkg

# Robustly find the notebook directory (handles VS Code, Jupyter, terminal)
NOTEBOOK_DIR = let
    d = @__DIR__
    if !isempty(d) && isdir(d)
        d
    elseif isfile(joinpath(pwd(), "Example_1.ipynb"))
        pwd()
    elseif isfile(joinpath(pwd(), "Example_1", "Example_1.ipynb"))
        joinpath(pwd(), "Example_1")
    else
        @warn "Could not detect notebook directory. Using pwd(): $(pwd())"
        pwd()
    end
end

Pkg.activate(NOTEBOOK_DIR)

# If Project.toml is missing or empty, install all required packages
if !isfile(joinpath(NOTEBOOK_DIR, "Project.toml")) || isempty(Pkg.project().dependencies)
    Pkg.add([
        "JuMP", "HiGHS", "Ipopt", "Symbolics",
        "DifferentialEquations", "MathOptInterface",
        "LaTeXStrings", "PyPlot", "Combinatorics",
    ])
end

Pkg.instantiate()


In [ ]:
## Import Packages
using JuMP
using HiGHS
using Ipopt
using Symbolics
using LinearAlgebra
using DifferentialEquations
using MathOptInterface
using LaTeXStrings
using PyPlot
using Combinatorics

import PyPlot
const plt = PyPlot

In [ ]:
## Plot Settings
SMALL_SIZE = 7
MEDIUM_SIZE = 8
BIGGER_SIZE = 9

# Set all parameters using rc()
plt.rc("text", usetex=false)
plt.rc("font", family="Times New Roman", size=MEDIUM_SIZE)
plt.rc("mathtext", fontset="cm")
plt.rc("lines", linewidth=1)
plt.rc("legend", labelspacing=0.25, fontsize=MEDIUM_SIZE)
plt.rc("axes", titlesize=MEDIUM_SIZE, labelsize=MEDIUM_SIZE)
plt.rc("xtick", labelsize=SMALL_SIZE)
plt.rc("ytick", labelsize=SMALL_SIZE)
plt.rc("figure", titlesize=BIGGER_SIZE)

In [ ]:
## System and Simulation Parameters
t_span = (0, 30);               # Time span as a tuple
dt = 0.1;                       # Time step
x0 = [10.0, 0.0];                   # Initial condition as a vector
xf = [0.0, 0.0];                # Desired final state
umin = -1;                      # Minimum control input
umax = 1;                       # Maximum control input
vmin = -1;                      # Minimum velocity
vmax = 1;                       # Maximum velocity

Q = I(2);                       # State cost weight
R = 1.0;                        # Control cost weight

In [ ]:
## Hovercraft Dynamics (for DifferentialEquations.jl)
function hovercraft_plant(x, p, t)
    # Extract control from parameters (p is a vector)
    u = p[1]  # Extract scalar control
    
    # States
    # x[1] = position
    # x[2] = velocity
    
    # Dynamics
    fx = [x[2], 
          0.0]
    gx = [0.0, 
          1.0]
    
    xdot = fx + gx * u
    return xdot
end

In [ ]:
## =========================================================================
## SGA Policy Iteration for GHJBCBF Controller 
## =========================================================================

# N-dimensional polynomial structure
struct PolyND{N}
    coeffs::Dict{NTuple{N,Int},Real}
end
# constructor: using just coeffs, size implied
function PolyND(coeffs::Dict{NTuple{N,Int64},T}) where {N,T <: Real}
    return PolyND{N}(coeffs)
end
# constructor: empty zero polynomial in N dims
PolyND{N}() where {N} = PolyND{N}(Dict{NTuple{N,Int},Float64}())
# constructor: using Symbolics object
function PolyND(expr::Num)
    poly = Symbolics.expand(expr)
    vars = Symbolics.get_variables(poly)
    N = length(vars)
    coeffs = Dict{NTuple{N,Int},Real}()
    @inbounds for term in terms(poly)
        exponents = Tuple( Symbolics.degree(term,var) for var in vars )
        coeffs[exponents] = Symbolics.substitute(term,Dict(var => 1 for var in vars))
    end
    return PolyND{N}(coeffs)
end
# constructor: using Symbolics object and specified variable list
function PolyND(expr::Num, vars::Vector{Num})
    poly = Symbolics.expand(expr)
    N = length(vars)
    coeffs = Dict{NTuple{N,Int},Real}()
    @inbounds for term in terms(poly)
        exponents = Tuple( Symbolics.degree(term,var) for var in vars )
        coeffs[exponents] = Symbolics.substitute(term,Dict(var => 1 for var in vars))
    end
    return PolyND{N}(coeffs)
end
# Evaluate p at x = (x1, x2, …, xN)
function (p::PolyND{N})(x::NTuple{N,Real}) where {N}
    s = zero(Float64)
    @inbounds for (α, c) in p.coeffs
        # α is an N-tuple of exponents
        prod = c
        @inbounds for k in 1:N
            prod *= x[k]^α[k]
        end
        s += prod
    end
    return s
end
# overloaded for individual arguments
function (p::PolyND{N})(xs::Vararg{Real,N}) where {N}
    # xs is a Tuple{Real,Real,…} of length N, so
    # just forward to the NTuple method:
    p(xs)
end
function integrate(p::PolyND{N}, bounds::NTuple{N,Tuple{<:Real,<:Real}}) where {N}
    total = 0.0
    @inbounds for (α, c) in p.coeffs
        term = c
        for k in 1:N
            ℓ, u = bounds[k]
            m = α[k]
            term *= (u^(m+1) - ℓ^(m+1)) / (m+1)
        end
        total += term
    end
    return total
end
function run_SGA_policy_iteration(;
    # Basis function parameters
    N_Phi::Int = 3,                    # Number of basis functions to use
    max_iter::Int = 15,                # Maximum SGA iterations
    
    # Cost function parameters
    Q = Q,                             # State cost weight
    R = R,                             # Control cost weight
    
    # Integration domain
    bounds = [(-1.0, 1.0), 
              (-1.0, 1.0)],            # State space bounds for integration
    
    # Numerical parameters
    rtol = 1e-8,                       # Relative tolerance for integration
    atol = 1e-10,                      # Absolute tolerance for integration
    maxevals = 0,                      # Max evaluations (0 = automatic)
    
    # Convergence parameters
    V_diff_tol = 1e-3,                 # Convergence tolerance on coefficient change
    
    # Options
    verbose = true,                    # Print iteration progress
)
    # Define state variables
    @Symbolics.variables x1, x2, u
    # States
    x = [x1, x2]
    num_states = length(x)
    # Control inputs
    u = u
    # Define the basis functions as polynomials
    Phi_all = [x1^2, x2^2, x1*x2]
    # Select subset of basis functions
    Phi = Phi_all[1:N_Phi]
    # Number of basis functions
    n = length(Phi)
    # Define system dynamics
    fx = [x2, 0.0]
    gx = [0.0, 1.0]
    # Define nonlinear control affine form
    xdot = fx + gx * u
    # Define the running state cost function q(x)
    q = x' * Q * x
    # Initialize the value function and optimal control
    V_opt = []
    u_opt = []
    c_opt = []
    # Define unknown coefficients c_sym = [c0, c1, ..., cn-1]
    c_sym = [Symbolics.variable(:c, i) for i in 1:n]
    push!(c_opt, c_sym)
    # Define the polynomial approximation of V(x)
    V0 = Phi' * c_sym
    gradV0_x = Symbolics.gradient(V0, x)
    # Initial control (a stabilizing guess)
    u_opt0 = -x1 - x2  # Stabilizing linear control
    push!(u_opt, u_opt0)
    # Extract bounds for hcubature
    lb = Float64[b[1] for b in bounds]  # lower bounds
    ub = Float64[b[2] for b in bounds]  # upper bounds
    # Galerkin approximation main loop
    for iter in 1:max_iter
        H = gradV0_x' * (fx + gx * u_opt[iter]) + q + R * u_opt[iter]^2
        # Compute A(t) = ∫ Φ * (∂H/∂c_j) dΩ, keeping t symbolic
        Ai = Phi * Symbolics.jacobian([H], c_sym)  # ∂H/∂c includes dc/dt terms
        # convert to fast poly objects and integrate analytically
        Afast = [ PolyND( Ai[i,j], [x1,x2] ) for i in 1:n, j in 1:n ]
        A = [ integrate( Afast[i,j], Tuple(zip(lb,ub)) ) for i in 1:n, j in 1:n ]
        # Compute b(t) = ∫ Φ * (H with c derivatives zeroed) dΩ
        bi = - Phi * substitute(H, Dict(c => 0 for c in c_sym))  # Zero out dc/dt and c(t) terms
        # convert to fast poly object and integrate analytically
        bfast = [ PolyND( bi[i], [x1, x2] ) for i in 1:n ]
        b = [ integrate( bfast[i], Tuple(zip(lb,ub)) ) for i in 1:n ]
        
        # Solve for coefficients
        c_opt_new = Float64.(Symbolics.value.(A \ b))
        push!(c_opt, c_opt_new)
        
        # Update value function
        V_new = Phi' * c_opt_new                       # Scalar value function
        push!(V_opt, V_new)
        
        # Update control
        gradV_new = Symbolics.gradient(V_new, x)
        u_opt_new = - (1/2) * R * gx' * gradV_new      # u* = -1/2 R^-1 g^T ∇V
        push!(u_opt, u_opt_new)

        if verbose
            println("\n----------------- Iteration $iter -----------------")
            println("Number of basis functions: $n")
            println("Max |c|: ", maximum(abs.(c_opt_new)))
        end

        # Convergence check (coefficient norm)
        if iter > 1
            c_prev = Float64.(Symbolics.value.(c_opt[iter]))  # previous coefficients (offset by 1 due to initial c_sym push)
            c_diff = norm(c_opt_new - c_prev)

            if verbose
                println("Change in c coefficients: $c_diff")
            end

            if iter == max_iter
                if verbose
                    println("\nMaximum iterations reached\n")
                end
                return (
                    u_opt = u_opt_new,
                    V_opt = V_new,
                    c_opt = c_opt_new,
                    u_opt_history = u_opt,
                    V_opt_history = V_opt,
                    c_opt_history = c_opt,
                    Phi = Phi,
                    iterations = iter,
                )
            end

            if c_diff < V_diff_tol
                if verbose
                    println("\nConverged after $iter iterations\n")
                end
                return (
                    u_opt = u_opt_new,
                    V_opt = V_new,
                    c_opt = c_opt_new,
                    u_opt_history = u_opt,
                    V_opt_history = V_opt,
                    c_opt_history = c_opt,
                    Phi = Phi,
                    iterations = iter,
                )
            end
        end
    end
end

@time results = run_SGA_policy_iteration();

In [ ]:
## =========================================================================
## GHJBCBF Controller 
## =========================================================================

function GHJBCBF_control(x, alpha, R_weight;
                        results = results,
                        umin, umax,
                        vmin, vmax)
    # Define symbolic state variables
    Symbolics.@variables x1 x2
    # Compute static V from converged SGA results
    V_sym = (results.Phi' * results.c_opt)[1]
    # Compute the gradient of V with respect to state variables
    gradV_sym = Symbolics.gradient(V_sym, [x1, x2])
    gradV_num = Symbolics.substitute(gradV_sym, Dict(x1 => x[1], x2 => x[2]))
    gradV_num = Float64.(Symbolics.value.(gradV_num))
    # System dynamics
    gx = [0.0, 1.0]
    # Compute ∇V' * g(x)
    LgV = dot(gradV_num, gx)
    # Control barrier functions for velocity constraints
    B_unbar = x[2] - vmin   # h_lower(x) = v - v_min  (>0 when safe)
    B_bar   = vmax - x[2]   # h_upper(x) = v_max - v  (>0 when safe)
    # Set up QP optimization problem
    model = Model(HiGHS.Optimizer)
    set_silent(model)
    JuMP.@variable(model, u)
    # Objective: minimize deviation from optimal + control effort
    #   min  ∇V'g · u + R · u²
    JuMP.@objective(model, Min,
        LgV * u + R_weight * u^2
    )
    # CBF constraints: enforce velocity bounds
    JuMP.@constraint(model, u >= -alpha * B_unbar)   # lower velocity barrier
    JuMP.@constraint(model, u <=  alpha * B_bar)     # upper velocity barrier
    # Hard control input bounds
    JuMP.@constraint(model, u >= umin)
    JuMP.@constraint(model, u <= umax)
    # Solve QP
    optimize!(model)
    # Extract solution
    u_val = value(u)
    return u_val
end

In [ ]:
## ==========================================================================
## GHJB-CBF-QP Simulation
## ==========================================================================

alpha = 10.0

R_scalar = R

t_grid = range(t_span[1], t_span[2], step=dt)
n_t_grid = length(t_grid)
u_GHJBCBF = zeros(1, n_t_grid)
x_GHJBCBF = zeros(2, n_t_grid)
x_GHJBCBF[:, 1] = x0

start_time = time()
for i in 1:(n_t_grid-1)
    u_GHJBCBF[1, i] = GHJBCBF_control(x_GHJBCBF[:, i], alpha, R_scalar;
                              results=results,
                              umin=umin, umax=umax, vmin=vmin, vmax=vmax)

    t_temp = (t_grid[i], t_grid[i] + dt)
    prob = ODEProblem(hovercraft_plant, x_GHJBCBF[:, i], t_temp, [u_GHJBCBF[1, i]])
    sol = solve(prob, RK4(), saveat=t_temp, reltol=1e-4, abstol=1e-6)
    x_GHJBCBF[:, i+1] = sol.u[end]
end
elapsed_GHJBCBF = time() - start_time
t_GHJBCBF = collect(t_grid)

# Cost: ∫₀ᵀ (x'Qx + u'Ru) dt 
Cost_GHJBCBF = sum(
    (x_GHJBCBF[:, i]' * Q * x_GHJBCBF[:, i] + u_GHJBCBF[:, i]' * R * u_GHJBCBF[:, i]) * dt
    for i in 1:n_t_grid-1
)

println("GHJBCBF elapsed time: $(round(elapsed_GHJBCBF, digits=2)) s")
println("GHJBCBF cost J = $Cost_GHJBCBF")

v_threshold = 0.01
if any((vmax - v_threshold) .<= x_GHJBCBF[2, :] .<= (vmax + v_threshold)) ||
   any((vmin - v_threshold) .<= x_GHJBCBF[2, :] .<= (vmin + v_threshold))
    println("GHJBCBF: velocity constraint active")
end
if any(u_GHJBCBF[:] .>= umax) || any(u_GHJBCBF[:] .<= umin)
    println("GHJBCBF: input constraint active")
end

In [ ]:
## ==========================================================================
## Nonlinear Optimal Control (RK2 Dynamics)
## ==========================================================================
Tf = t_span[2]
n_steps = Int(Tf / dt)
timepts = range(0, Tf, length=n_steps+1)

model_ocp = Model(Ipopt.Optimizer)
set_optimizer_attribute(model_ocp, "max_iter", 1000)
set_optimizer_attribute(model_ocp, "tol", 1e-10)
set_optimizer_attribute(model_ocp, "mu_strategy", "adaptive")
set_optimizer_attribute(model_ocp, "print_level", 1)

@variable(model_ocp, X[1:2, 1:n_steps+1])
@variable(model_ocp, U[1:1, 1:n_steps+1])

# Explicit Midpoint (RK2) dynamics constraints
for t in 1:n_steps
    x_t = X[:, t]
    u_t = U[:, t]
    
    # Stage 1: evaluate at current state
    k1 = hovercraft_plant(x_t, u_t, timepts[t])
    
    # Stage 2: evaluate at midpoint
    x_mid = x_t + (dt/2) * k1
    k2 = hovercraft_plant(x_mid, u_t, timepts[t] + 0.5*dt)
    
    # Midpoint update
    @constraint(model_ocp, X[:, t+1] .== x_t + dt * k2)
end

# Running cost only
cost = sum(
    (X[:, t] - xf)' * Q * (X[:, t] - xf) + U[:, t]' * R * U[:, t]
    for t in 1:n_steps
) * dt

@objective(model_ocp, Min, cost)
@constraint(model_ocp, X[:, 1] .== x0)
@constraint(model_ocp, [t=1:n_steps+1], -20 <= X[1, t] <= 20)
@constraint(model_ocp, [t=1:n_steps+1], vmin <= X[2, t] <= vmax)
@constraint(model_ocp, [t=1:n_steps+1], umin <= U[1, t] <= umax)

start_time = time()
optimize!(model_ocp)
elapsed_OCP = time() - start_time

x_OCP = value.(X)
u_OCP = value.(U)
t_OCP = collect(timepts)
Cost_OCP = objective_value(model_ocp)

println("OCP elapsed time: $(round(elapsed_OCP, digits=2)) s")
println("OCP cost J = $Cost_OCP")

v_threshold = 0.01
if any((vmax - v_threshold) .<= x_OCP[2, :] .<= (vmax + v_threshold)) ||
   any((vmin - v_threshold) .<= x_OCP[2, :] .<= (vmin + v_threshold))
    println("OCP: velocity constraint active")
end

if any(u_OCP[:] .>= umax) || any(u_OCP[:] .<= umin)
    println("OCP: input constraint active")
end

In [ ]:
## ==========================================================================
## MPC with Infinite-Time LQR Terminal Cost
## ==========================================================================
## MPC parameters
N_mpc = 20                # Prediction horizon (steps)
dt_mpc = dt               # Same time step as simulation

## Compute P_inf from CARE
F_lin = [0.0 1.0; 0.0 0.0]
G_lin = reshape([0.0; 1.0], 2, 1)

function solve_care_mpc(F, G, Q, R)
    R_inv = inv(R)
    H = [F        -G * R_inv * G';
        -Q        -F']
    eig = eigen(H)
    stable_idx = findall(real.(eig.values) .< 0)
    n = size(F, 1)
    if length(stable_idx) != n
        error("Expected $n stable eigenvalues, found $(length(stable_idx))")
    end
    V_stable = eig.vectors[:, stable_idx]
    U1 = V_stable[1:n, :]
    U2 = V_stable[n+1:end, :]
    P = real.(U2 / U1)
    return P
end

P_inf_mpc = solve_care_mpc(F_lin, G_lin, Q, R)

function MPC_control(x_current, P_inf, Q_in, R_in, N_mpc, dt_mpc;
                     umin=-1.0, umax=1.0, vmin=-0.5, vmax=0.5)
    n_x = 2
    n_u = 1

    # Ensure Q is a Float64 matrix and R is a scalar Float64
    Q_mat = Float64.(Matrix(isa(Q_in, AbstractMatrix) ? Q_in : Q_in * I(n_x)))
    R_val = Float64(isa(R_in, AbstractMatrix) ? R_in[1,1] : R_in)

    # Linearized dynamics: x_{k+1} = A_d x_k + B_d u_k  (Euler discretization)
    A_d = [1.0  dt_mpc;
           0.0  1.0]
    B_d = [0.0;
           dt_mpc]

    # Set up QP
    model = Model(Ipopt.Optimizer)
    set_silent(model)
    set_optimizer_attribute(model, "tol", 1e-8)
    set_optimizer_attribute(model, "max_iter", 500)

    @variable(model, X[1:n_x, 1:N_mpc+1])
    @variable(model, uvar[1:N_mpc])

    # Initial condition
    @constraint(model, X[:, 1] .== x_current)

    # Dynamics constraints
    for k in 1:N_mpc
        @constraint(model, X[:, k+1] .== A_d * X[:, k] + B_d * uvar[k])
    end

    # State constraints: velocity bounds at all steps
    for k in 1:N_mpc+1
        @constraint(model, X[2, k] >= vmin)
        @constraint(model, X[2, k] <= vmax)
        @constraint(model, X[1, k] >= -20.0)
        @constraint(model, X[1, k] <= 20.0)
    end

    # Input constraints
    for k in 1:N_mpc
        @constraint(model, uvar[k] >= umin)
        @constraint(model, uvar[k] <= umax)
    end

    # Objective: stage cost + terminal cost
    stage_cost = sum(
        X[:, k]' * Q_mat * X[:, k] * dt_mpc + R_val * uvar[k]^2 * dt_mpc
        for k in 1:N_mpc
    )
    terminal_cost = X[:, N_mpc+1]' * P_inf * X[:, N_mpc+1]

    @objective(model, Min, stage_cost + terminal_cost)

    optimize!(model)

    # Check solve status
    status = termination_status(model)
    if status != MOI.OPTIMAL && status != MOI.LOCALLY_SOLVED
        @warn "MPC solver status: $status at x = $x_current. Using fallback u = 0."
        return 0.0
    end

    # Return first control action (receding horizon)
    return value(uvar[1])
end

## Simulate MPC
u_MPC = zeros(1, n_t_grid)
x_MPC = zeros(2, n_t_grid)
x_MPC[:, 1] = x0

start_time = time()
for i in 1:(n_t_grid-1)
    u_MPC[1, i] = MPC_control(x_MPC[:, i], P_inf_mpc, Q, R, N_mpc, dt_mpc;
                               umin=umin, umax=umax, vmin=vmin, vmax=vmax)
    t_temp = (t_grid[i], t_grid[i] + dt)
    prob = ODEProblem(hovercraft_plant, x_MPC[:, i], t_temp, [u_MPC[1, i]])
    sol = solve(prob, RK4(), saveat=t_temp, reltol=1e-4, abstol=1e-6)
    x_MPC[:, i+1] = sol.u[end]
end
elapsed_MPC = time() - start_time
t_MPC = collect(t_grid)

Cost_MPC = sum(
    (x_MPC[:, i]' * Float64.(Matrix(Q)) * x_MPC[:, i] +
     Float64(isa(R, AbstractMatrix) ? R[1,1] : R) * u_MPC[1, i]^2) * dt
    for i in 1:n_t_grid-1
)
println("MPC elapsed time: $(round(elapsed_MPC, digits=2)) s")
println("MPC cost J = $Cost_MPC")

v_threshold = 0.01
if any((vmax - v_threshold) .<= x_MPC[2, :] .<= (vmax + v_threshold)) ||
   any((vmin - v_threshold) .<= x_MPC[2, :] .<= (vmin + v_threshold))
    println("MPC: velocity constraint active")
end

if any(u_MPC[:] .>= umax) || any(u_MPC[:] .<= umin)
    println("MPC: input constraint active")
end

In [ ]:
## ==========================================================================
## Constrained LQR (Multiparametric QP)
## ==========================================================================

## Compute P_inf from CARE
F_lin = [0.0 1.0; 0.0 0.0]
G_lin = reshape([0.0; 1.0], 2, 1)

function solve_care_clqr(F, G, Q, R)
    R_inv = inv(R)
    H = [F        -G * R_inv * G';
        -Q        -F']
    eig = eigen(H)
    stable_idx = findall(real.(eig.values) .< 0)
    n = size(F, 1)
    if length(stable_idx) != n
        error("Expected $n stable eigenvalues, found $(length(stable_idx))")
    end
    V_stable = eig.vectors[:, stable_idx]
    U1 = V_stable[1:n, :]
    U2 = V_stable[n+1:end, :]
    P = real.(U2 / U1)
    return P
end

P_inf_clqr = solve_care_clqr(F_lin, G_lin, Q, R)

## -------------------------------------------------------------------------
## Step 1: Build the condensed QP matrices for a single-step horizon
## -------------------------------------------------------------------------
function build_constrained_lqr(P_inf, Q_in, R_in, dt;
                               umin=-1.0, umax=1.0,
                               vmin=-0.5, vmax=0.5)
    R_val = Float64(isa(R_in, AbstractMatrix) ? R_in[1,1] : R_in)

    A_d = [1.0  dt;
           0.0  1.0]
    B_d = [0.0; dt]

    H_qp = (R_val * dt + B_d' * P_inf * B_d)[1]
    F_qp = A_d' * P_inf * B_d

    G_qp = [1.0; -1.0; dt; -dt]
    w_qp = [umax; -umin; vmax; -vmin]
    E_qp = [0.0 0.0;
            0.0 0.0;
            0.0 -1.0;
            0.0  1.0]

    return H_qp, F_qp, G_qp, w_qp, E_qp, A_d, B_d
end

## -------------------------------------------------------------------------
## Step 2: Enumerate active sets and solve KKT for each
## -------------------------------------------------------------------------
struct ConstrainedLQRRegion
    K::Vector{Float64}
    k::Float64
    C::Matrix{Float64}
    d::Vector{Float64}
end

function compute_constrained_lqr_regions(P_inf, Q, R, dt;
                                         umin=-1.0, umax=1.0,
                                         vmin=-0.5, vmax=0.5)
    H_qp, F_qp, G_qp, w_qp, E_qp, _, _ = build_constrained_lqr(
        P_inf, Q, R, dt; umin=umin, umax=umax, vmin=vmin, vmax=vmax)

    n_constraints = length(G_qp)
    all_indices = collect(1:n_constraints)
    regions = ConstrainedLQRRegion[]

    for n_active in 0:n_constraints
        for active_set in combinations(all_indices, n_active)
            inactive_set = setdiff(all_indices, active_set)

            if isempty(active_set)
                K_region = -F_qp / H_qp
                k_region = 0.0
                C_region = G_qp * K_region' - E_qp
                d_region = w_qp
            else
                G_A = G_qp[active_set]
                w_A = w_qp[active_set]
                E_A = E_qp[active_set, :]

                if length(active_set) > 1
                    continue
                end

                g_a = G_A[1]
                w_a = w_A[1]
                e_a = E_A[1, :]

                K_region = e_a / g_a
                k_region = w_a / g_a

                lambda_K = -(H_qp * K_region' + F_qp') / g_a
                lambda_k = -(H_qp * k_region) / g_a

                G_I = G_qp[inactive_set]
                w_I = w_qp[inactive_set]
                E_I = E_qp[inactive_set, :]

                C_inactive = G_I * K_region' - E_I
                d_inactive = w_I - G_I * k_region

                C_dual = -lambda_K
                d_dual = [lambda_k]

                C_region = vcat(C_inactive, C_dual)
                d_region = vcat(d_inactive, d_dual)
            end

            region = ConstrainedLQRRegion(vec(K_region), k_region,
                                          Matrix(C_region), vec(d_region))
            push!(regions, region)
        end
    end

    return regions
end

## -------------------------------------------------------------------------
## Step 3: Online evaluation — find region and apply affine law
## -------------------------------------------------------------------------
function constrained_lqr_eval(x, regions::Vector{ConstrainedLQRRegion})
    for region in regions
        if all(region.C * x .<= region.d .+ 1e-10)
            return dot(region.K, x) + region.k
        end
    end
    return dot(regions[1].K, x) + regions[1].k
end

## -------------------------------------------------------------------------
## Step 4: Precompute regions (offline) and simulate (online)
## -------------------------------------------------------------------------

@time regions = compute_constrained_lqr_regions(P_inf_clqr, Q, R, dt;
                                                umin=umin, umax=umax,
                                                vmin=vmin, vmax=vmax)

## Simulate
u_CLQR = zeros(1, n_t_grid)
x_CLQR = zeros(2, n_t_grid)
x_CLQR[:, 1] = x0

start_time = time()
for i in 1:(n_t_grid-1)
    u_CLQR[1, i] = constrained_lqr_eval(x_CLQR[:, i], regions)
    t_temp = (t_grid[i], t_grid[i] + dt)
    prob = ODEProblem(hovercraft_plant, x_CLQR[:, i], t_temp, [u_CLQR[1, i]])
    sol = solve(prob, RK4(), saveat=t_temp, reltol=1e-4, abstol=1e-6)
    x_CLQR[:, i+1] = sol.u[end]
end
elapsed_CLQR = time() - start_time
t_CLQR = collect(t_grid)

R_cost = Float64(isa(R, AbstractMatrix) ? R[1,1] : R)
Q_cost = Float64.(Matrix(isa(Q, AbstractMatrix) ? Q : Q * I(2)))
Cost_CLQR = sum(
    (x_CLQR[:, i]' * Q_cost * x_CLQR[:, i] + R_cost * u_CLQR[1, i]^2) * dt
    for i in 1:n_t_grid-1
)
println("\nConstrained LQR elapsed time: $(round(elapsed_CLQR, digits=4)) s")
println("Constrained LQR cost J = $Cost_CLQR")

v_threshold = 0.01
if any((vmax - v_threshold) .<= x_CLQR[2, :] .<= (vmax + v_threshold)) ||
   any((vmin - v_threshold) .<= x_CLQR[2, :] .<= (vmin + v_threshold))
    println("Constrained LQR: velocity constraint active")
end

if any(u_CLQR[:] .>= umax) || any(u_CLQR[:] .<= umin)
    println("Constrained LQR: input constraint active")
end

In [ ]:
## ==========================================================================
## Plot Trajectories (Position, Velocity, Control) — Hovercraft
## ==========================================================================
# Requires: 
#           x_CLQR, u_CLQR, t_CLQR                (Constrained LQR)
#           x_MPC, u_MPC, t_MPC                   (MPC)
#           x_OCP, u_OCP, t_OCP                   (Optimal Control)
#           x_GHJBCBF, u_GHJBCBF, t_GHJBCBF       (GHJB-CBF-QP)

mkpath("Example_1_Figures")

## ============================================
## Controller data dictionary
## ============================================
selected_controllers = ["1", "2", "3", "4"]
input_list = ["1", "2", "3", "4"]

controllers_data = Dict(
    "1" => Dict(
        "label" => "Constrained LQR",
        "color" => "tab:green",
        "x" => x_CLQR,
        "u" => u_CLQR,
        "t" => t_CLQR,
        "linestyle" => "solid"
    ),
    "2" => Dict(
        "label" => "MPC",
        "color" => "tab:orange",
        "x" => x_MPC,
        "u" => u_MPC,
        "t" => t_MPC,
        "linestyle" => "solid"
    ),
    "3" => Dict(
        "label" => "Open-Loop Optimal (OCP)",
        "color" => "tab:blue",
        "x" => x_OCP,
        "u" => u_OCP,
        "t" => t_OCP,
        "linestyle" => "solid"
    ),
    "4" => Dict(
        "label" => "GHJB-CBF-QP",
        "color" => "black",
        "x" => x_GHJBCBF,
        "u" => u_GHJBCBF,
        "t" => t_GHJBCBF,
        "linestyle" => "solid"
    )
)

## ============================================
## Plot 1: State and Control Trajectories
## ============================================
fig, (ax1, ax2, ax3) = subplots(3, figsize=(3.41, 3.2), sharex=true)
fig.set_constrained_layout(true)

for controller in selected_controllers
    if controller in input_list
        data = controllers_data[controller]
        ax1.plot(data["t"], data["x"][1, :],
                 linestyle=data["linestyle"], color=data["color"],
                 zorder=3, label=data["label"])
        ax2.plot(data["t"], data["x"][2, :],
                 linestyle=data["linestyle"], color=data["color"],
                 zorder=3)
        ax3.plot(data["t"], data["u"][1, :],
                 linestyle=data["linestyle"], color=data["color"],
                 zorder=3)
    end
end

# Position axis
ax1.hlines(y=0, xmin=0, xmax=t_grid[end], color="grey", linestyle="dotted", zorder=2)
ax1.set_ylabel(L"p \text{ (m)}")

# Velocity axis with constraint bounds
ax2.hlines(y=0, xmin=0, xmax=t_grid[end], color="grey", linestyle="dotted", zorder=2)
ax2.hlines(y=vmax, xmin=0, xmax=t_grid[end], color="red", linestyle="dashed", zorder=2, label="Input/State Constraints")
ax2.hlines(y=vmin, xmin=0, xmax=t_grid[end], color="red", linestyle="dashed", zorder=2)
ax2.set_ylabel(L"v \text{ (m/s)}")

# Control axis with input bounds
ax3.hlines(y=0, xmin=0, xmax=t_grid[end], color="grey", linestyle="dotted", zorder=2)
ax3.hlines(y=umax, xmin=0, xmax=t_grid[end], color="red", linestyle="dashed", zorder=2)
ax3.hlines(y=umin, xmin=0, xmax=t_grid[end], color="red", linestyle="dashed", zorder=2)
ax3.set_ylabel(L"u \text{ (N)}")

# Grid and axis formatting
for ax in [ax1, ax2, ax3]
    ax.autoscale(enable=true, axis="x", tight=true)
    ax.grid(true, zorder=0, color="0.95", linestyle="dotted", which="both", axis="both")
end

ax3.set_xlabel("Time (s)")

# ============================================
# Fill to padded plot edges
# ============================================
yl2 = ax2.get_ylim()
pad2 = 0.05 * (yl2[2] - yl2[1])  # 5% padding — adjust to taste
new_yl2 = (min(yl2[1], vmin) - pad2, max(yl2[2], vmax) + pad2)
ax2.fill_between(collect(t_grid), vmax, new_yl2[2], alpha=0.2, color="red")
ax2.fill_between(collect(t_grid), new_yl2[1], vmin, alpha=0.2, color="red")
ax2.set_ylim(new_yl2)

yl3 = ax3.get_ylim()
pad3 = 0.05 * (yl3[2] - yl3[1])  # 5% padding — adjust to taste
new_yl3 = (min(yl3[1], umin) - pad3, max(yl3[2], umax) + pad3)
ax3.fill_between(collect(t_grid), umax, new_yl3[2], alpha=0.2, color="red")
ax3.fill_between(collect(t_grid), new_yl3[1], umin, alpha=0.2, color="red")
ax3.set_ylim(new_yl3)
# ============================================

# Combined legend
handles1, labels1 = ax1.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()
handles3, labels3 = ax3.get_legend_handles_labels()
handles = vcat(handles1, handles2, handles3)
labels = vcat(labels1, labels2, labels3)

if length(handles) > 0
    fig.legend(handles, labels, loc="outside upper center", ncol=2,
               framealpha=0, edgecolor="none", facecolor="white").set_zorder(2)
end

# show()
savefig("Example_1_Figures/trajectories_hovercraft.pdf", dpi=300, bbox_inches="tight", pad_inches=0.0)
println("Saved: Example_1_Figures/trajectories_hovercraft.pdf")